In [1]:
# 1. Install required packages
# Run this once in your environment
# pip install kaggle pandas sqlite3

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
# %pip install seaborn

In [2]:
print(os.getcwd())

/Users/mac2025/Desktop/Continuous Education/Scaler/Sessions/Portfolio_projects/BeginnerSQLProject


In [3]:
# 3. Load CSV into pandas
df_ops = pd.read_csv('pos_operator_logs.csv')
df_tx = pd.read_csv('pos_transactions.csv')

In [4]:
df_ops.head()

,id,Workstation_Group_ID,Workstation_ID,begin_date_time,operator_id
0,1,8,19,2019-02-13T05:37:55,269
1,2,8,18,2019-02-13T05:37:55,268
2,3,8,17,2019-02-13T05:38:43,267
3,4,1,4,2019-02-13T07:01:26,332
4,5,1,7,2019-02-13T07:01:57,10


In [5]:
df_tx.head()

,id,WorkstationGroupID,begin_date_time,end_date_time,OperatorID,basket_size,t_cash,t_card,amount
0,1,1,2017-12-07T14:23:23,2017-12-07T14:24:36,101,23,True,False,112.71
1,2,1,2017-12-07T14:25:09,2017-12-07T14:27:00,101,29,True,False,54.76
2,3,1,2017-12-07T14:27:28,2017-12-07T14:27:48,101,3,True,False,14.77
3,4,1,2017-12-07T14:28:04,2017-12-07T14:28:29,101,12,True,False,37.88
4,5,1,2017-12-07T14:29:40,2017-12-07T14:30:32,101,7,True,False,115.34


In [6]:
df_ops.shape

(9396, 5)

In [7]:
df_tx.shape

(163269, 9)

In [8]:
conn2 = sqlite3.connect('transaction_data.db')
df_tx.to_sql('transaction_data', conn2, index=False, if_exists='replace')

163269

In [9]:
def query_and_print(sql):
    result = pd.read_sql(sql, conn2)
    print(result, '\n')

analysis_queries = {
    'Task 1: Do more people make transactions by card or by cash?': '''
        SELECT 
        SUM(CASE WHEN t_cash = 1 THEN 1 ELSE 0 END) AS cash_transactions,
        SUM(CASE WHEN t_card = 1 THEN 1 ELSE 0 END) AS card_transactions
        FROM transaction_data
    ''',
    
    'Task 2: Do people spend more per transaction when using cash or card?': '''
        SELECT 
        ROUND( AVG(CASE WHEN t_cash = 1 THEN amount END) , 2) AS Avg_cash,
        ROUND( AVG(CASE WHEN t_card = 1 THEN amount END) , 2) AS Avg_card
        FROM transaction_data
    ''',
    
    'Task 3: Working and Non-Working Sundays?': '''
        SELECT
        strftime('%W', end_date_time) AS week_num,
        end_date_time AS end_date
        FROM transaction_data 
        WHERE strftime('%Y', end_date_time) >= '2019'
        AND 
        strftime('%Y-%m-%d', end_date_time) IN ('2019-02-17', '2019-02-24', '2019-03-31', '2019-04-07')
        GROUP BY end_date;
     ''',
    
    'Task 4: Daily Trends': '''
    SELECT
    strftime('%W', end_date_time) AS week_num,
    end_date_time AS end_date,
    COUNT (id) AS total_transactions,
    SUM(amount) AS total_sales,
    AVG(amount) AS avg_sale_amount
    FROM transaction_data
    WHERE strftime('%Y', end_date_time) >= '2019'
    GROUP BY end_date
    ORDER BY week_num;
    ''',
    
    'Task 5: Market Basket Size': '''
    WITH OrderedAmounts AS (
      SELECT
        id,
        end_date_time,
        basket_size,
        amount,
        strftime('%W', end_date_time) AS week_num,
        ROW_NUMBER() OVER (PARTITION BY strftime('%W', end_date_time) ORDER BY amount) AS rn_amount,
        COUNT(*) OVER (PARTITION BY strftime('%W', end_date_time)) AS cnt_amount,
        ROW_NUMBER() OVER (PARTITION BY strftime('%W', end_date_time) ORDER BY basket_size) AS rn_basket,
        COUNT(*) OVER (PARTITION BY strftime('%W', end_date_time)) AS cnt_basket
      FROM transaction_data
      WHERE strftime('%Y', end_date_time) >= '2019'
    ),
    Medians AS (
      SELECT
        week_num,
        AVG(CASE WHEN rn_amount IN ((cnt_amount + 1)/2, (cnt_amount + 2)/2) THEN amount END) AS median_sale_amount,
        AVG(CASE WHEN rn_basket IN ((cnt_basket + 1)/2, (cnt_basket + 2)/2) THEN basket_size END) AS median_basket_size
      FROM OrderedAmounts
      GROUP BY week_num
    )
    SELECT
      oa.week_num,
      MIN(oa.end_date_time) AS end_date,  -- or use MAX depending on need
      COUNT(oa.id) AS total_transactions,
      SUM(oa.amount) AS total_sales,
      AVG(oa.amount) AS avg_sale_amount,
      m.median_sale_amount,
      AVG(oa.basket_size) AS avg_basket_size,
      m.median_basket_size
    FROM OrderedAmounts oa
    JOIN Medians m ON oa.week_num = m.week_num
    GROUP BY oa.week_num
    ORDER BY oa.week_num;

    '''
}

In [10]:
for desc, sql in analysis_queries.items():
    print(f'-- {desc} --')
    query_and_print(sql)

conn2.close()

-- Task 1: Do more people make transactions by card or by cash? --
   cash_transactions  card_transactions
0              84487              78246 

-- Task 2: Do people spend more per transaction when using cash or card? --
   Avg_cash  Avg_card
0     59.62     87.56 

-- Task 3: Working and Non-Working Sundays? --
     week_num             end_date
0          07  2019-02-24T06:42:35
1          07  2019-02-24T06:42:42
2          07  2019-02-24T06:42:59
3          07  2019-02-24T06:43:29
4          07  2019-02-24T06:43:38
...       ...                  ...
4474       12  2019-03-31T21:22:56
4475       12  2019-03-31T21:25:52
4476       12  2019-03-31T21:27:59
4477       12  2019-03-31T21:33:11
4478       12  2019-03-31T21:34:54

[4479 rows x 2 columns] 

-- Task 4: Daily Trends --
      week_num             end_date  total_transactions  total_sales  \
0           06  2019-02-13T06:38:23                   1         3.88   
1           06  2019-02-13T06:42:47                   1         

#### Working and Non-Working Sundays
- The dataset contains two working Sundays (24 February, 31 March) and two non-working Sundays (17 February, 7 April).

In [11]:
conn1 = sqlite3.connect('operator_data.db')
df_ops.to_sql('operator_data', conn1, index=False, if_exists='replace')

9396

In [12]:
sql = {
    'Task 6: Labor Costs': '''
    SELECT 
        strftime('%W', begin_date_time) AS working_day,
        strftime('%W', begin_date_time) AS week_num,
        COUNT(DISTINCT operator_id) AS distinct_operators
    FROM operator_data
    WHERE strftime('%Y', begin_date_time) >= '2019'
    GROUP BY working_day, week_num
    '''
}

In [13]:
result = pd.read_sql(sql['Task 6: Labor Costs'], conn1)
print(result, '\n')

  working_day week_num  distinct_operators
0          06       06                  37
1          07       07                  37
2          08       08                  23
3          12       12                  35
4          13       13                  43
5          14       14                  30 



In [14]:
conn1.close()